# ВКР

Ноутбук для интерактивного анализа. Весь производственный код вынесен в пакет `project/`.
Здесь — только вызовы функций, отображение результатов и исследовательский EDA.

In [ ]:
# ─── Настройка путей ──────────────────────────────────────────────────────────
import sys, os
# Добавляем корень проекта (директория project/) в путь
sys.path.insert(0, os.path.abspath(".."))

import logging
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s")


## 1. Конфигурация

Измените пути и горизонты при необходимости.

In [ ]:
from config import CFG

CFG["FILE_PATH"] = "../OZON_combined.csv"      # путь к CSV-файлу с котировками
CFG["HORIZONS"]  = [1, 5, 10]                  # горизонты прогнозирования (торг. дни)
CFG["OUT_DIR"]   = "../results"                # директория для артефактов

print("CFG загружен. Горизонты:", CFG["HORIZONS"])


## 2. Smoke Test

Быстрая проверка загрузки и формата данных.

In [ ]:
from main import smoke_test
smoke_test(CFG["FILE_PATH"])


## 3. Полный эксперимент

`run_experiment` запускает пайплайн: загрузка → EDA → **ARIMA–GARCH** (rolling μ̂, GARCH на ε̂) → ML. В словаре результатов таблица baseline: `results["arima_garch"]` (колонки `y_realized`, `mu_forecast`, `ci_lower`, `ci_upper`).

In [ ]:
from pipeline import run_experiment
results = run_experiment(CFG, CFG["FILE_PATH"], CFG["OUT_DIR"])


## 4. Walk-Forward: регрессия

In [ ]:
import pandas as pd
df_reg = pd.DataFrame(results.wf_reg_metrics)
display(
    df_reg.groupby(["model", "horizon"])[["MAE", "RMSE", "sMAPE", "MDA_%", "R2"]]
          .mean().round(4)
)


## 5. Walk-Forward: классификация

In [ ]:
df_clf = pd.DataFrame(results.wf_clf_metrics)
display(
    df_clf.groupby(["model", "horizon"])[["Accuracy", "F1_macro", "DirAcc_%", "Sharpe"]]
          .mean().round(4)
)


## 6. Бенчмарки: Buy & Hold и Naive

In [ ]:
print("Naive (r=0):")
for k, v in results.baseline_metrics.items():
    print(f"  {k}: {v}")

print("\nBuy & Hold:")
for k, v in results.buy_hold.items():
    print(f"  {k}: {v}")

print("\nStacking:")
for k, v in results.stacking_metrics.items():
    print(f"  {k}: {v}")


## 7. Предсказания и диагностика остатков

In [ ]:
if results.wf_predictions is not None and not results.wf_predictions.empty:
    display(results.wf_predictions.head(10))
    print(f"Предсказаний: {len(results.wf_predictions)} строк")
else:
    print("Предсказания недоступны.")


## 8. Optuna — лучшие гиперпараметры

In [ ]:
print("Optuna best params (h=1):")
for k, v in results.optuna_params.items():
    print(f"  {k}: {v}")
